#### Reading PDF Data using LLMs

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Source Folder Path
SRC_FOLDER = "/Volumes/atci_databricks_hackathon_2025/atci_clinical_transcript_ai_hackathon/raw_data/"

In [0]:
%sql
--select * from `system.ai.databricks-gemini-3-pro`;

SELECT ai_query(
  "databricks-meta-llama-3-3-70b-instruct",
  "Hello! Can you hear me? Respond with the word SUCCESS."
) as test_result



In [0]:
# Prompt for LLMs to extract specific fields
PROMPT = """
You are a clinical data extractor.
Extract the following fields in JSON format:

patient_name
mrn
dob
visit_date
provider
reason
subjective
objective
bp systolic
bp diastolic
Heart rate
assessment & plan

TEXT:
{note}
"""

In [0]:
# Extracting fields from parsed text
columns = ['file_name',
 'page_number',
 'patient_name',
 'mrn',
 'dob',
 'visit_date',
 'provider',
 'reason',
 'subjective',
 'objective',
 'bp_systolic',
 'bp_diastolic',
 'heart_rate',
 'assessment_and_plan']

schema = StructType([
    StructField("patient_name", StringType()),
    StructField("mrn", StringType()),
    StructField("dob", StringType()),
    StructField("visit_date", StringType()),
    StructField("provider", StringType()),
    StructField("reason", StringType()),
    StructField("subjective", StringType()),
    StructField("objective", StringType()),
    StructField("bp_systolic", IntegerType()),
    StructField("bp_diastolic", IntegerType()),
    StructField("heart_rate", IntegerType()),
    StructField("assessment_and_plan", StringType())
])

bronze_table_df = spark.table("atci_databricks_hackathon_2025.atci_clinical_transcript_ai_hackathon.clinial_data_extracted_bronze")

df_structured = bronze_table_df.withColumn("llm_json", expr(f"""
      AI_QUERY(
        'databricks-meta-llama-3-3-70b-instruct',
        CONCAT('{PROMPT}', page_text)
      )"""))

In [0]:
df_structured.printSchema()

In [0]:
    df_structured = df_structured.withColumn(
            "llm_json_clean",
            trim(
                regexp_extract(
                    col("llm_json"),
                    r"(?s)(\{.*\})",
                    1
                )
            )
        )

    df_structured = df_structured.withColumn("extracted_json", from_json(col("llm_json_clean"), schema,{"mode": "PERMISSIVE"}))

    df_final = df_structured.select("file_name", "page_number", "page_text", "llm_json", *[
            col(f"extracted_json.{field.name}").alias(field.name)
            for field in schema.fields
        ])

In [0]:
display(df_final)

In [0]:
# Writing data to another silver with LLM table
df_final.write.mode("overwrite").saveAsTable("atci_databricks_hackathon_2025.atci_clinical_transcript_ai_hackathon.clinial_data_LLM_parsed_silver")

In [0]:
%sql
select count(*) from atci_databricks_hackathon_2025.atci_clinical_transcript_ai_hackathon.Clinial_data_LLM_parsed_silver where patient_name is null;


In [0]:
Next_visit_Prompt = """
You are a clinical assistant.

TASK:
From the ASSESSMENT_AND_PLAN text, determine the number of days until the next visit and Drugs Prescribed in the text.

RULES:
- Only extract if explicitly stated (e.g. "return in 4 weeks", "follow up after 2 months")
- Convert weeks/months to DAYS:
  - 1 week = 7 days
  - 1 month = 30 days
- If unclear or not stated, return null
- Extract the Drug name with dosage (e.g. "Lisinopril 10mg","Amlodipine 5mg")
- Return JSON ONLY, no explanation

OUTPUT:
{ "days_until_next_visit": <integer or null> ,
"drug_name_with_dosage": <string or null>}

ASSESSMENT_AND_PLAN:
{{assessment_and_plan}}
"""

days_schema = StructType([
    StructField("days_until_next_visit", IntegerType()),
    StructField("drug_name_with_dosage", StringType())
])

In [0]:
# Extracting days until next visit
df_final=df_final.withColumn("followup_json", expr(f"""
      AI_QUERY(
        'databricks-meta-llama-3-3-70b-instruct',
        CONCAT('{Next_visit_Prompt}', assessment_and_plan)
      )"""))


df_days = df_final.withColumn(
    "days_struct",
    from_json(col("followup_json"), days_schema, {"mode": "PERMISSIVE"})
)

In [0]:
# Adding Next Visit Date
df_final_with_next_visit_date = (
    df_days
    .withColumn(
        "next_visit_date",
        date_add(
            col("visit_date"),
            col("days_struct.days_until_next_visit")
        )
    )
    .withColumn(
        "drug_name",
        col("days_struct.drug_name_with_dosage")
    )
    .drop("days_struct")
)

In [0]:
display(df_final_with_next_visit_date)

In [0]:
df_final_with_next_visit_date.printSchema()

In [0]:
# Writing data to another silver with LLM table
df_final_with_next_visit_date.write.mode("overwrite").saveAsTable("atci_databricks_hackathon_2025.atci_clinical_transcript_ai_hackathon.clinial_data_LLM_parsed_gold")